In [5]:
import os, sys, json, math, random
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F

def parse_paths():
    if len(sys.argv) > 2 and not sys.argv[1].startswith('-f'):
        pub, out = Path(sys.argv[1]), Path(sys.argv[2])
    else:
        candidates = ['/kaggle/input/datasets/devansh66/urban-surface-marking',
                      '/kaggle/input/urban-surface-marking',
                      '/kaggle/input', './dataset/public', '.']
        pub = next((Path(c) for c in candidates if (Path(c) / 'train.csv').exists()), Path('./dataset/public'))
        out = Path('./working/submission.csv')
    os.makedirs(out.parent, exist_ok=True)
    return pub, out

PUB, OUT = parse_paths()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEV == 'cuda'
torch.backends.cudnn.benchmark = True
R, C = 9, 12
IH, IW = 288, 192          # 432x288 tested: no gain, 2.25x slower
CH, CW = IH//R, IW//C      # pixels per grid cell (32 x 16)
SEEDS = [42, 1337]         # spend the saved time on ensembling instead
FOLDS, EPOCHS, BS = 5, 50, 40
NCNT, NAREA, NHIST, NVB = 31, 9, 10, 10
NCX, NCY, NW, NH = 8, 8, 7, 7
SM = ('small', 'medium', 'large')
CELLB = [6, 12, 20, 30, 45]

def tok(r, c): return "r%02d_c%02d" % (r, c)

def components(g):
    lab = -np.ones((R, C), int); k = 0
    for i in range(R):
        for j in range(C):
            if g[i, j] and lab[i, j] < 0:
                st = [(i, j)]; lab[i, j] = k
                while st:
                    a, b = st.pop()
                    for da, db in ((1,0),(-1,0),(0,1),(0,-1)):
                        x, y = a+da, b+db
                        if 0 <= x < R and 0 <= y < C and g[x, y] and lab[x, y] < 0:
                            lab[x, y] = k; st.append((x, y))
                k += 1
    return lab, k

def f1(p, t):
    P, T = set(p), set(t)
    if not P and not T: return 1.0
    if not P or not T: return 0.0
    i = len(P & T)
    return 0.0 if i == 0 else 2*i/(len(P)+len(T))

def dupf1(p, t):
    P = Counter(tuple(sorted(d.items())) for d in p); T = Counter(tuple(sorted(d.items())) for d in t)
    if not P and not T: return 1.0
    if not P or not T: return 0.0
    i = sum((P & T).values())
    return 0.0 if i == 0 else 2*i/(sum(P.values())+sum(T.values()))

def parts(p, t):
    return dict(
        count=max(0, 1-abs(p['marking_count']-t['marking_count'])/8),
        occ=f1(p['occupied_cells'], t['occupied_cells']),
        edge=f1(p['edge_touch_cells'], t['edge_touch_cells']),
        hist=sum(max(0, 1-abs(p['size_histogram'][k]-t['size_histogram'][k])/4) for k in SM)/3,
        vert=sum(max(0, 1-abs(a-b)/5) for a, b in zip(p['vertical_band_counts'], t['vertical_band_counts']))/9,
        area=max(0, 1-abs(p['area_bin']-t['area_bin'])/4),
        card=dupf1(p['region_cards'], t['region_cards']))

W = dict(count=.14, occ=.27, edge=.12, hist=.12, vert=.12, area=.08, card=.15)
def row_score(p, t):
    q = parts(p, t); return sum(W[k]*q[k] for k in W)

class Calib:
    def __init__(self, A):
        self.area = self._lut([(len(a['occupied_cells']), a['area_bin']) for a in A], R*C)
        pr = []
        for a in A:
            if a['marking_count'] == 1:
                k = [k for k, v in a['size_histogram'].items() if v == 1]
                if k: pr.append((len(a['occupied_cells']), SM.index(k[0])))
        self.sz = self._lut(pr, R*C)
        px, py, pw, ph = [], [], [], []
        for a in A:
            if a['marking_count'] == 1 and len(a['region_cards']) == 1:
                cs = a['occupied_cells']
                r = [int(c[1:3]) for c in cs]; c_ = [int(c[5:7]) for c in cs]
                d = a['region_cards'][0]
                px.append((min(c_)+max(c_), d['cx_bin'])); py.append((min(r)+max(r), d['cy_bin']))
                pw.append((max(c_)-min(c_)+1, d['w_bin'])); ph.append((max(r)-min(r)+1, d['h_bin']))
        self.cx = self._lut(px, 2*C); self.cy = self._lut(py, 2*R)
        self.wb = self._lut(pw, C); self.hb = self._lut(ph, R)
        self.cnt = None

    @staticmethod
    def _lut(pairs, maxk):
        d = defaultdict(Counter)
        for k, v in pairs: d[k][v] += 1
        lut = {k: d[k].most_common(1)[0][0] for k in d}
        ks = sorted(lut)
        return [lut[k] if k in lut else (lut[min(ks, key=lambda x: abs(x-k))] if ks else 0)
                for k in range(maxk+1)]

    def derive(self, g):
        g = g.astype(bool)
        occ = [tok(i, j) for i in range(R) for j in range(C) if g[i, j]]
        lab, k = components(g)
        comps = []
        for t in range(k):
            rs, cs = np.where(lab == t)
            comps.append((int(rs.size), rs.min(), rs.max(), cs.min(), cs.max()))
        hist = {'small': 0, 'medium': 0, 'large': 0}; vb = [0]*R
        for s, r0, r1, c0, c1 in comps:
            hist[SM[self.sz[min(s, R*C)]]] += 1
            vb[int(round((r0+r1)/2))] += 1
        comps.sort(key=lambda x: -x[0])
        cards = [{'cx_bin': int(self.cx[c0+c1]), 'cy_bin': int(self.cy[r0+r1]),
                  'w_bin': int(self.wb[c1-c0+1]), 'h_bin': int(self.hb[r1-r0+1])}
                 for s, r0, r1, c0, c1 in comps[:12]]
        ncomp = len(comps)
        cnt = ncomp
        if self.cnt is not None:
            cnt = self.cnt.get((ncomp, int(np.digitize(len(occ), CELLB))), max(1, ncomp))
        return {'marking_count': int(min(max(cnt, 1), 30)), 'occupied_cells': occ,
                'edge_touch_cells': [], 'size_histogram': {k: min(v, 9) for k, v in hist.items()},
                'vertical_band_counts': [min(v, 9) for v in vb],
                'area_bin': int(self.area[min(len(occ), R*C)]),
                'region_cards': cards, '_ncomp': ncomp}

def load_imgs(df):
    X = np.zeros((len(df), IH, IW, 3), np.uint8)
    for i, p in enumerate(df.image_path.values):
        fp = PUB / p
        if not fp.exists(): fp = PUB / 'images' / Path(p).name
        X[i] = np.asarray(Image.open(fp).convert('RGB').resize((IW, IH), Image.BILINEAR))
    return X

train = pd.read_csv(PUB / 'train.csv'); test = pd.read_csv(PUB / 'test.csv')
A = [json.loads(x) for x in train.answer_json]
cal = Calib(A); N = len(A)
Y = np.zeros((N, R, C), np.float32)
Tcnt = np.zeros(N, np.int64); Tarea = np.zeros(N, np.int64)
Thist = np.zeros((N, 3), np.int64); Tvb = np.zeros((N, R), np.int64)
Tcard = np.zeros((N, 4), np.int64); Mcard = np.zeros(N, np.float32)
for i, a in enumerate(A):
    for c in a['occupied_cells']: Y[i, int(c[1:3]), int(c[5:7])] = 1
    Tcnt[i] = min(a['marking_count'], NCNT-1); Tarea[i] = a['area_bin']
    Thist[i] = [min(a['size_histogram'][k], NHIST-1) for k in SM]
    Tvb[i] = [min(v, NVB-1) for v in a['vertical_band_counts']]
    if a['marking_count'] == 1 and len(a['region_cards']) == 1:
        d = a['region_cards'][0]; Mcard[i] = 1
        Tcard[i] = [min(d['cx_bin'], NCX-1), min(d['cy_bin'], NCY-1),
                    min(d['w_bin'], NW-1), min(d['h_bin'], NH-1)]
Xtr = load_imgs(train); Xte = load_imgs(test)
print('images', Xtr.shape, Xte.shape, flush=True)

def block(ci, co, s=2):
    return nn.Sequential(nn.Conv2d(ci, co, 3, s, 1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
                         nn.Conv2d(co, co, 3, 1, 1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True))

HEADS = [('cnt', NCNT), ('area', NAREA), ('h0', NHIST), ('h1', NHIST), ('h2', NHIST),
         ('cx', NCX), ('cy', NCY), ('w', NW), ('h', NH)] + [('v%d' % i, NVB) for i in range(R)]

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.b = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 192))
        self.ctx = nn.Sequential(nn.Conv2d(192, 192, 3, 1, 2, dilation=2, bias=False),
                                 nn.BatchNorm2d(192), nn.ReLU(inplace=True), nn.Dropout2d(0.2))
        self.occ = nn.Conv2d(192, 1, 1)
        self.trunk = nn.Sequential(nn.Linear(192*2, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3))
        self.hs = nn.ModuleDict({k: nn.Linear(256, n) for k, n in HEADS})
    def forward(self, x):
        f = self.ctx(self.b(x))
        o = self.occ(F.adaptive_avg_pool2d(f, (R, C)))[:, 0]
        # DETACHED: head gradients never touch the backbone -> occupancy trains exactly as in v1
        d = f.detach()
        g = torch.cat([F.adaptive_avg_pool2d(d, 1).flatten(1), F.adaptive_max_pool2d(d, 1).flatten(1)], 1)
        g = self.trunk(g)
        return o, {k: self.hs[k](g) for k, _ in HEADS}

MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 3, 1, 1)
def norm(u8):
    return (u8.permute(0, 3, 1, 2).float()/255. - MEAN) / STD

FLIP_CX = torch.tensor([NCX-1-i for i in range(NCX)], device=DEV)

def predict(model, X, bs=32):
    model.eval(); O = []; H = {k: [] for k, _ in HEADS}
    with torch.no_grad(), torch.autocast('cuda', enabled=AMP):
        for i in range(0, len(X), bs):
            xb = norm(torch.from_numpy(X[i:i+bs]).to(DEV)).contiguous(memory_format=torch.channels_last)
            o1, h1 = model(xb); o2, h2 = model(torch.flip(xb, [3]))
            O.append(((torch.sigmoid(o1.float()) + torch.flip(torch.sigmoid(o2.float()), [2]))/2).cpu().numpy())
            for k, _ in HEADS:
                p1 = F.softmax(h1[k].float(), 1); p2 = F.softmax(h2[k].float(), 1)
                if k == 'cx': p2 = p2[:, FLIP_CX]
                H[k].append(((p1+p2)/2).cpu().numpy())
    return np.concatenate(O), {k: np.concatenate(v) for k, v in H.items()}

def wmed(p):
    return int(np.argmax(np.cumsum(p) >= 0.5))

idx = np.arange(N); rng = np.random.RandomState(SEED); rng.shuffle(idx)
folds = np.array_split(idx, FOLDS)
oof_o = np.zeros((N, R, C), np.float32)
oof_h = {k: np.zeros((N, n), np.float32) for k, n in HEADS}
te_o = np.zeros((len(test), R, C), np.float32)
te_h = {k: np.zeros((len(test), n), np.float32) for k, n in HEADS}
pw = torch.tensor((1-Y.mean())/Y.mean()*0.6).to(DEV)
HW = {'cnt': .3, 'area': .2, 'h0': .15, 'h1': .15, 'h2': .15,
      'cx': .2, 'cy': .2, 'w': .2, 'h': .2}
for i in range(R): HW['v%d' % i] = .04

Tg = {'cnt': Tcnt, 'area': Tarea, 'h0': Thist[:, 0], 'h1': Thist[:, 1], 'h2': Thist[:, 2],
      'cx': Tcard[:, 0], 'cy': Tcard[:, 1], 'w': Tcard[:, 2], 'h': Tcard[:, 3]}
for i in range(R): Tg['v%d' % i] = Tvb[:, i]

def shift_cells(xb, yb, dy, dx):
    """Translate image by a whole number of grid cells and translate the label grid
    identically. Cells scrolled in from outside have unknown labels -> mask weight 0."""
    xb = torch.roll(xb, (dy*CH, dx*CW), (2, 3))
    yb = torch.roll(yb, (dy, dx), (1, 2))
    m = torch.ones_like(yb)
    if dy > 0: m[:, :dy, :] = 0
    elif dy < 0: m[:, dy:, :] = 0
    if dx > 0: m[:, :, :dx] = 0
    elif dx < 0: m[:, :, dx:] = 0
    return xb, yb, m

RUNS = len(SEEDS)*FOLDS
for seed in SEEDS:
  random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
  for f in range(FOLDS):
    va = folds[f]; tr = np.concatenate([folds[j] for j in range(FOLDS) if j != f])
    model = Net().to(DEV).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    steps = EPOCHS * max(1, math.ceil(len(tr)/BS))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    Xf = torch.from_numpy(Xtr[tr]).to(DEV)                  # uint8 on GPU, cheap
    Yf = torch.from_numpy(Y[tr]).to(DEV)
    Tf = {k: torch.from_numpy(np.asarray(v)[tr]).to(DEV) for k, v in Tg.items()}
    Mf = torch.from_numpy(Mcard[tr]).to(DEV)
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(tr), device=DEV)
        for i in range(0, len(tr), BS):
            b = perm[i:i+BS]
            if b.numel() < 2: continue
            xb = norm(Xf[b]); yb = Yf[b].clone(); flipped = False
            if random.random() < 0.5:
                xb = torch.flip(xb, [3]); yb = torch.flip(yb, [2]); flipped = True
            wm = None
            if random.random() < 0.6:                       # cell-aligned translation
                dy, dx = random.randint(-1, 1), random.randint(-2, 2)
                if dy or dx: xb, yb, wm = shift_cells(xb, yb, dy, dx)
            if random.random() < 0.8:
                r = lambda: torch.rand(xb.size(0), 1, 1, 1, device=DEV)
                xb = xb*(0.8 + 0.4*r()) + 0.3*(r()-0.5)
            if random.random() < 0.3:
                xb = xb + 0.05*torch.randn_like(xb)
            xb = xb.contiguous(memory_format=torch.channels_last)
            with torch.autocast('cuda', enabled=AMP):
                o, hd = model(xb)
                if wm is None:
                    occ_loss = F.binary_cross_entropy_with_logits(o, yb, pos_weight=pw)
                else:
                    e = F.binary_cross_entropy_with_logits(o, yb, pos_weight=pw, reduction='none')
                    occ_loss = (e*wm).sum()/wm.sum().clamp(min=1)
                hloss = torch.zeros((), device=DEV)
                if wm is None:          # a shift invalidates the global targets
                    for k, n in HEADS:
                        t = Tf[k][b]
                        if k == 'cx' and flipped: t = (NCX-1) - t
                        ce = F.cross_entropy(hd[k], t, reduction='none', label_smoothing=0.05)
                        if k in ('cx', 'cy', 'w', 'h'):
                            m = Mf[b]; ce = (ce*m).sum()/m.sum().clamp(min=1)
                        else:
                            ce = ce.mean()
                        hloss = hloss + HW[k]*ce
                loss = occ_loss + hloss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    o, h = predict(model, Xtr[va]); oof_o[va] += o/len(SEEDS)
    for k in h: oof_h[k][va] += h[k]/len(SEEDS)
    o, h = predict(model, Xte); te_o += o/RUNS
    for k in h: te_h[k] += h[k]/RUNS
    print('seed %d fold %d  occ_loss %.4f' % (seed, f, occ_loss.item()), flush=True)

# ---------- grid builder: guarantees non-empty ----------
def make_grid(prob, th, kmin=3):
    g = prob >= np.float32(th)
    if g.sum() < kmin:                       # tie-safe top-k, never returns the whole grid
        g = np.zeros(R*C, bool)
        g[np.argpartition(-prob.ravel(), kmin-1)[:kmin]] = True
        g = g.reshape(R, C)
    return g

def assemble(prob, h, i, th, use):
    b = cal.derive(make_grid(prob[i], th))
    if use['cnt']: b['marking_count'] = max(1, wmed(h['cnt'][i]))
    if use['area']: b['area_bin'] = wmed(h['area'][i])
    if use['hist']: b['size_histogram'] = {k: wmed(h['h%d' % j][i]) for j, k in enumerate(SM)}
    if use['vb']: b['vertical_band_counts'] = [wmed(h['v%d' % j][i]) for j in range(R)]
    if use['card'] and len(b['region_cards']) == 1:
        b['region_cards'] = [{'cx_bin': int(h['cx'][i].argmax()), 'cy_bin': int(h['cy'][i].argmax()),
                              'w_bin': int(h['w'][i].argmax()), 'h_bin': int(h['h'][i].argmax())}]
    return b

def ev(th, use, detail=False):
    B = [assemble(oof_o, oof_h, i, th, use) for i in range(N)]
    Q = [parts(b, a) for b, a in zip(B, A)]
    s = float(np.mean([sum(W[k]*q[k] for k in W) for q in Q]))
    if detail:
        return s, {k: float(np.mean([q[k] for q in Q])) for k in W}, B
    return s

use = {k: False for k in ('cnt', 'area', 'hist', 'vb', 'card')}
best, bth = -1, .5
for th in np.arange(.20, .80, .02):
    s = ev(th, use)
    if s > best: best, bth = s, th
print('\nbaseline (all derived)  OOF %.4f @ th=%.2f' % (best, bth), flush=True)

# LUTs were fitted on TRUE grids but are applied to PREDICTED ones, which are
# systematically smaller. Refit them on the OOF grids to remove that bias.
def refit(th):
    c = Calib(A)
    pa, ps, px, py, pw_, ph = [], [], [], [], [], []
    for n, a in enumerate(A):
        d = c.derive(make_grid(oof_o[n], th)); occ = d['occupied_cells']
        if not occ: continue
        pa.append((len(occ), a['area_bin']))
        if a['marking_count'] == 1:
            k = [k for k, v in a['size_histogram'].items() if v == 1]
            if k: ps.append((len(occ), SM.index(k[0])))
            if len(a['region_cards']) == 1 and d['_ncomp'] == 1:
                r = [int(x[1:3]) for x in occ]; cc = [int(x[5:7]) for x in occ]
                cd = a['region_cards'][0]
                px.append((min(cc)+max(cc), cd['cx_bin'])); py.append((min(r)+max(r), cd['cy_bin']))
                pw_.append((max(cc)-min(cc)+1, cd['w_bin'])); ph.append((max(r)-min(r)+1, cd['h_bin']))
    c.area = c._lut(pa, R*C); c.sz = c._lut(ps, R*C)
    c.cx = c._lut(px, 2*C); c.cy = c._lut(py, 2*R)
    c.wb = c._lut(pw_, C); c.hb = c._lut(ph, R)
    return c
old = cal; cal = refit(bth); s = ev(bth, use)
if s > best:
    print('LUT refit on OOF grids  ON  %.4f (+%.4f)' % (s, s-best), flush=True); best = s
else:
    print('LUT refit on OOF grids  off %.4f' % s, flush=True); cal = old

# component-count -> marking_count remap, fitted on OOF (components under-count badly)
dv = [cal.derive(make_grid(oof_o[i], bth)) for i in range(N)]
ncomp = np.array([x['_ncomp'] for x in dv])
cbin = np.array([int(np.digitize(len(x['occupied_cells']), CELLB)) for x in dv])
tc = np.array([a['marking_count'] for a in A])
lut = {}
for k in range(ncomp.max()+1):
    for j in range(len(CELLB)+1):
        m = (ncomp == k) & (cbin == j)
        if m.sum() >= 8: lut[(k, j)] = int(np.median(tc[m]))
cal.cnt = lut
s = ev(bth, use)
if s > best:
    print('count remap ON  %.4f (+%.4f)  cells=%d' % (s, s-best, len(lut)), flush=True); best = s
else:
    print('count remap off %.4f' % s, flush=True); cal.cnt = None

for k in ('cnt', 'area', 'hist', 'vb', 'card'):
    use[k] = True; s = ev(bth, use)
    if s > best + 1e-5:
        print('  head[%-4s] ON  %.4f (+%.4f)' % (k, s, s-best), flush=True); best = s
    else:
        print('  head[%-4s] off %.4f' % (k, s), flush=True); use[k] = False
for th in np.arange(.20, .80, .02):
    s = ev(th, use)
    if s > best: best, bth = s, th

s, det, B = ev(bth, use, detail=True)
print('\nFINAL OOF %.4f @ th=%.2f  heads=%s  count_remap=%s' %
      (s, bth, [k for k in use if use[k]], cal.cnt is not None), flush=True)

# ---------- diagnostics ----------
print('\n%-6s %8s %8s %8s %10s' % ('field', 'weight', 'score', 'oracle', 'pts lost'))
ORACLE = dict(count=.846, occ=1.0, edge=.800, hist=.867, vert=.957, area=.905, card=.182)
for k in ('occ', 'card', 'count', 'edge', 'hist', 'vert', 'area'):
    print('%-6s %8.2f %8.4f %8.3f %10.4f' % (k, W[k], det[k], ORACLE[k], W[k]*(ORACLE[k]-det[k])))

def dist(name, fn, pred, truth):
    a = np.array([fn(p) for p in pred]); b = np.array([fn(p) for p in truth])
    print('%-9s pred mean %6.2f med %3.0f max %3.0f | true mean %6.2f med %3.0f max %3.0f'
          % (name, a.mean(), np.median(a), a.max(), b.mean(), np.median(b), b.max()))
print('\ncalibration (OOF vs train truth):')
dist('ncells', lambda p: len(p['occupied_cells']), B, A)
dist('count', lambda p: p['marking_count'], B, A)
dist('area_bin', lambda p: p['area_bin'], B, A)
dist('ncards', lambda p: len(p['region_cards']), B, A)
print('empty grids (must be 0):', sum(1 for b in B if not b['occupied_cells']))

hp = np.zeros((R, C)); ht = np.zeros((R, C))
for b in B:
    for c in b['occupied_cells']: hp[int(c[1:3]), int(c[5:7])] += 1
for a in A:
    for c in a['occupied_cells']: ht[int(c[1:3]), int(c[5:7])] += 1
print('\nheatmap  pred - true  (large +ve = model over-predicting there):')
print(((hp-ht)/N).round(2))

rows = []
for i, rid in enumerate(test.id.values):
    b = assemble(te_o, te_h, i, bth, use); b.pop('_ncomp', None)
    rows.append({'id': rid, 'answer_json': json.dumps(b, separators=(',', ':'))})
sub = pd.DataFrame(rows)
assert len(sub) == len(test) and sub.id.duplicated().sum() == 0
assert all(json.loads(x)['occupied_cells'] for x in sub.answer_json), 'empty grid in submission'
sub.to_csv(OUT, index=False)
print('\nwrote', OUT, len(sub))

images (700, 288, 192, 3) (322, 288, 192, 3)
seed 42 fold 0  occ_loss 0.0465
seed 42 fold 1  occ_loss 0.0316
seed 42 fold 2  occ_loss 0.0446
seed 42 fold 3  occ_loss 0.0773
seed 42 fold 4  occ_loss 0.0547
seed 1337 fold 0  occ_loss 0.0926
seed 1337 fold 1  occ_loss 0.0487
seed 1337 fold 2  occ_loss 0.0403
seed 1337 fold 3  occ_loss 0.0696
seed 1337 fold 4  occ_loss 0.0378

baseline (all derived)  OOF 0.6426 @ th=0.36
LUT refit on OOF grids  ON  0.6453 (+0.0027)
count remap ON  0.6476 (+0.0023)  cells=18
  head[cnt ] off 0.6415
  head[area] off 0.6410
  head[hist] off 0.6444
  head[vb  ] off 0.6463
  head[card] off 0.6421

FINAL OOF 0.6476 @ th=0.36  heads=[]  count_remap=True

field    weight    score   oracle   pts lost
occ        0.27   0.5718    1.000     0.1156
card       0.15   0.0608    0.182     0.0182
count      0.14   0.8271    0.846     0.0026
edge       0.12   0.8000    0.800     0.0000
hist       0.12   0.8236    0.867     0.0052
vert       0.12   0.9388    0.957     0.0022